# 03 — Embedding
### DiscoverAI · Deloitte × LUISS

Builds dense vector representations for all products using a two-tower architecture.

**Architecture:**
```
product_text_base  →  MPNet  →  product_emb  (768-dim)
                                                        →  combined_emb  →  FAISS index
review texts       →  MPNet  →  review_emb   (768-dim)
```

**Key design choices:**
| Choice | Reason |
|---|---|
| all-mpnet-base-v2 | STS benchmark 72.4 vs 68.1 for MiniLM — better semantic quality |
| Two-tower | Respects MPNet's 512-token limit; product and review signals stay separable |
| Weighted review average | 70% of raw reviews are 5-star — plain average is biased |
| Dynamic alpha per product | Products with weak review signal → product embedding dominates |
| Bias correction | Rare reviews (e.g. negative reviews in mostly-positive products) get boosted |
| IndexFlatIP | Exact search; < 5ms for 11k products; no approximation needed |

**Output files (all in DATA_DIR):**
- `product_embeddings.npy` — (11537, 768)
- `review_embeddings.npy`  — (11537, 768)
- `combined_embeddings.npy` — (11537, 768)
- `faiss_index.bin` — FAISS IndexFlatIP
- `embedding_index.csv` — maps row index → product metadata


## 0 · Setup

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
import faiss

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

if torch.backends.mps.is_available():   DEVICE = "mps"
elif torch.cuda.is_available():          DEVICE = "cuda"
else:                                    DEVICE = "cpu"
print(f"Device: {DEVICE}")


## 1 · Configuration

Set `DATA_DIR` to your data folder. All other parameters are documented inline.
On Google Colab with T4 GPU, uncomment the Drive mount lines and update DATA_DIR.


In [ ]:
DATA_DIR      = "data"  # update this path if needed
PRODUCTS_PATH = os.path.join(DATA_DIR, "products_cleaned.csv")
REVIEWS_PATH  = os.path.join(DATA_DIR, "reviews_cleaned.csv")

MODEL_NAME = "all-mpnet-base-v2"
BATCH_SIZE = 64
MIN_TOKENS = 25

# Pesi review base
W_HELPFUL = 0.7
W_RATING  = 0.3

# Alpha dinamico: range [ALPHA_MIN, ALPHA_MAX]
# prodotti con segnale review debole → alpha alto (product emb pesa di più)
# prodotti con segnale review forte  → alpha basso (review emb pesa di più)
ALPHA_MIN = 0.35   # prodotti con tante review utili
ALPHA_MAX = 0.70   # prodotti con poche review o tutte senza helpful_vote

print("Config OK")
print(f"  Modello : {MODEL_NAME}")
print(f"  Alpha   : dinamico [{ALPHA_MIN}, {ALPHA_MAX}]")
print(f"  Pesi    : helpful={W_HELPFUL}, rating_dist={W_RATING} + bias correction")


## 2 · Load data

In [ ]:
products = pd.read_csv(PRODUCTS_PATH)
reviews  = pd.read_csv(REVIEWS_PATH)

reviews = reviews[reviews["tok_len"] >= MIN_TOKENS].reset_index(drop=True)

common   = set(products["parent_asin"]) & set(reviews["parent_asin"])
products = products[products["parent_asin"].isin(common)].reset_index(drop=True)
reviews  = reviews[reviews["parent_asin"].isin(common)].reset_index(drop=True)

print(f"Prodotti : {len(products):,}")
print(f"Review   : {len(reviews):,}")

asin_to_idx = {asin: i for i, asin in enumerate(products["parent_asin"])}


## 3 · Load model

In [ ]:
print(f"Loading model: {MODEL_NAME}")
t0    = time.time()
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
DIM   = model.get_sentence_embedding_dimension()
print(f"Ready in {time.time()-t0:.1f}s  |  embedding dim={DIM}")


## 4 · Product embedding

Encodes `product_text_base` for all 11,537 products.
`normalize_embeddings=True` produces L2-normalised vectors so that
inner product equals cosine similarity — required for FAISS IndexFlatIP.


In [ ]:
print("Encoding products...")
t0 = time.time()
product_emb = model.encode(
    products["product_text_base"].fillna("").astype(str).tolist(),
    batch_size=BATCH_SIZE, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True
)
print(f"\nFatto in {time.time()-t0:.1f}s  |  shape: {product_emb.shape}")
print(f"Norma media: {np.linalg.norm(product_emb, axis=1).mean():.4f}")


## 5 · Review embedding — weighted average

**Step A:** encode each review individually  
**Step B:** compute per-review weights  
**Step C:** compute weighted average per product

**Weight formula:**
```
weight_i = 0.7 × log1p(helpful_vote_i) / max(log1p) 
         + 0.3 × |rating_i - 3| / 2
         × bias_correction_i
```

- `helpful_vote` component: community-validated quality signal. log1p prevents
  a review with 5000 votes from dominating all others.
- `rating distance` component: reviews at rating extremes (1 or 5) carry more
  information than neutral reviews (3 stars).
- `bias_correction`: reviews that are rare for their product (e.g. a negative
  review in a mostly-positive product) get a boost factor of up to 3.0×.


In [ ]:
print("Encoding individual reviews...")
t0 = time.time()
review_emb_raw = model.encode(
    reviews["text"].fillna("").astype(str).tolist(),
    batch_size=BATCH_SIZE, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True
)
print(f"Done in {time.time()-t0:.1f}s  |  shape: {review_emb_raw.shape}")


In [ ]:
# Base weights ────────────────────────────────────────────────────────────────
hv          = reviews["helpful_vote"].fillna(0).values.astype(float)
hv_log      = np.log1p(hv)
hv_norm     = hv_log / hv_log.max() if hv_log.max() > 0 else hv_log
rating_dist = (np.abs(reviews["rating"].values - 3) / 2).astype(float)
weights_base = W_HELPFUL * hv_norm + W_RATING * rating_dist
weights_base = np.clip(weights_base, 1e-6, None)

# Bias correction: boost rare reviews within each product ───────────────────────────────────────────────────────────
# Per ogni prodotto: quanto è rara ogni review rispetto alla distribuzione globale?
# Review negative in prodotti positivi → peso amplificato
# Distribuzione globale attesa (dal cleaning v3)
global_dist = reviews["rating"].value_counts(normalize=True).to_dict()

# For each product, compare local vs global rating distribution
prod_dist = reviews.groupby("parent_asin")["rating"].value_counts(normalize=True)

bias_weights = np.ones(len(reviews))
for i, (_, row) in enumerate(reviews.iterrows()):
    asin   = row["parent_asin"]
    rating = row["rating"]
    local_pct  = prod_dist.get((asin, rating), 0.01)
    global_pct = global_dist.get(rating, 0.01)
    # If a review is locally rarer than globally expected → amplify
    # Clamp boost between 0.5 and 3.0 to avoid extreme amplification
    bias_weights[i] = np.clip(global_pct / max(local_pct, 0.01), 0.5, 3.0)

weights_final = weights_base * bias_weights
weights_final = np.clip(weights_final, 1e-6, None)

print(f"Final weights — min: {weights_final.min():.4f}  max: {weights_final.max():.4f}  media: {weights_final.mean():.4f}")

# Weight distribution plots
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
fig.suptitle("Distribuzione pesi review (con bias correction)", fontweight="bold")
axes[0].hist(hv_norm, bins=50, color="steelblue"); axes[0].set_title("helpful_vote norm")
axes[1].hist(bias_weights, bins=30, color="coral"); axes[1].set_title("bias correction factor")
axes[2].hist(weights_final, bins=50, color="seagreen"); axes[2].set_title("peso finale")
plt.tight_layout(); plt.show()


In [ ]:
# Compute weighted average per product ────────────────────────────────────────────────
review_emb = np.zeros((len(products), DIM), dtype=np.float32)

for asin, grp in reviews.groupby("parent_asin"):
    idx = asin_to_idx.get(asin)
    if idx is None: continue
    row_idxs = grp.index.tolist()
    embs = review_emb_raw[row_idxs]
    w    = weights_final[row_idxs]
    review_emb[idx] = (embs * w[:, None]).sum(axis=0) / w.sum()

norms      = np.linalg.norm(review_emb, axis=1, keepdims=True)
norms      = np.where(norms == 0, 1, norms)
review_emb = review_emb / norms

print(f"Review embedding — shape: {review_emb.shape}")
print(f"Norma media: {np.linalg.norm(review_emb, axis=1).mean():.4f}")


## 6 · Dynamic alpha per product

Alpha controls the blend between product embedding and review embedding:
```
combined = alpha × product_emb + (1 - alpha) × review_emb
```

Alpha is computed per product based on review signal quality:
- **n_reviews** (normalised, 3→16): more reviews = stronger signal
- **max_helpful_vote** (normalised, 0→20): community validation

Products with `alpha > 0.65` have weak review signals → product text dominates.  
Products with `alpha < 0.45` have strong review signals → review embedding matters more.


In [ ]:
# Compute review quality statistics per product
rev_stats = reviews.groupby("parent_asin").agg(
    n_rev       = ("rating", "count"),
    max_helpful = ("helpful_vote", "max"),
).reset_index()
rev_stats["max_helpful"] = rev_stats["max_helpful"].fillna(0)

# Quality signal: combination of n_reviews and helpful_vote
# n_rev normalised: 3 reviews → 0.0, 16+ reviews → 1.0
n_rev_norm = (rev_stats["n_rev"].clip(upper=16) - 3) / 13

# helpful normalised: 0 votes → 0.0, 20+ votes → 1.0
hv_prod_norm = rev_stats["max_helpful"].clip(upper=20) / 20

# Combined quality signal [0, 1]
quality = 0.6 * n_rev_norm + 0.4 * hv_prod_norm

# Alpha: high quality signal → low alpha (review embedding matters more)
alphas = ALPHA_MAX - quality * (ALPHA_MAX - ALPHA_MIN)

# asin → alpha map
asin_alpha = dict(zip(rev_stats["parent_asin"], alphas))

print(f"Dynamic alpha per product:")
print(f"  media={alphas.mean():.3f}  min={alphas.min():.3f}  max={alphas.max():.3f}")
print(f"  Products with alpha < 0.45 (review dominates): {(alphas < 0.45).mean():.1%}")
print(f"  Products with alpha > 0.60 (product dominates):  {(alphas > 0.60).mean():.1%}")

# Distribuzione alpha
fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(alphas, bins=40, color="steelblue", edgecolor="white", linewidth=0.3)
ax.axvline(alphas.mean(), color="red", ls="--", lw=1.5, label=f"media={alphas.mean():.3f}")
ax.set(title="Distribuzione alpha dinamico per prodotto",
       xlabel="Alpha (alto = product emb pesa di più)", ylabel="# Prodotti")
ax.legend()
plt.tight_layout(); plt.show()


## 7 · Combined embedding

In [ ]:
# Use each product's own alpha for blending
alpha_vec = np.array([
    asin_alpha.get(asin, 0.5)
    for asin in products["parent_asin"]
], dtype=np.float32)[:, None]   # shape (N, 1) per broadcasting

combined_emb = alpha_vec * product_emb + (1 - alpha_vec) * review_emb
norms        = np.linalg.norm(combined_emb, axis=1, keepdims=True)
norms        = np.where(norms == 0, 1, norms)
combined_emb = (combined_emb / norms).astype(np.float32)

print(f"Combined embedding — shape: {combined_emb.shape}")
print(f"Mean norm: {np.linalg.norm(combined_emb, axis=1).mean():.4f}")


## 8 · Quality check

Two checks:
- **PCA 2D**: compress 768 dims to 2 for visualisation — coloured by avg rating
- **Cosine similarity between random pairs**: should be centred near 0,
  meaning products are well-separated in the vector space


In [ ]:
from sklearn.decomposition import PCA

SAMPLE = min(2000, len(combined_emb))
rng    = np.random.RandomState(42)
sidx   = rng.choice(len(combined_emb), SAMPLE, replace=False)
pca    = PCA(n_components=2, random_state=42)
emb_2d = pca.fit_transform(combined_emb[sidx])
ratings = products["product_avg_rating"].values[sidx]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Embedding quality check", fontweight="bold")

sc = axes[0].scatter(emb_2d[:,0], emb_2d[:,1], c=ratings,
                     cmap="RdYlGn", alpha=0.4, s=10, vmin=1, vmax=5)
plt.colorbar(sc, ax=axes[0], label="Avg rating")
axes[0].set_title(f"PCA 2D projection (n={SAMPLE})")
axes[0].set_xlabel(f"PC1 — {pca.explained_variance_ratio_[0]*100:.1f}% var")
axes[0].set_ylabel(f"PC2 — {pca.explained_variance_ratio_[1]*100:.1f}% var")

n_pairs = 5000
ia = rng.randint(0, len(combined_emb), n_pairs)
ib = rng.randint(0, len(combined_emb), n_pairs)
cos = (combined_emb[ia] * combined_emb[ib]).sum(axis=1)
axes[1].hist(cos, bins=60, color="steelblue", edgecolor="white", linewidth=0.3)
axes[1].axvline(cos.mean(), color="red", ls="--", lw=1.5, label=f"media={cos.mean():.3f}")
axes[1].set_title("Cosine similarity — random pairs")
axes[1].set_xlabel("Cosine similarity")
axes[1].legend()
plt.tight_layout(); plt.show()

print(f"Variance explained PC1+PC2: {pca.explained_variance_ratio_[:2].sum()*100:.1f}%")
print(f"Mean cosine similarity:   {cos.mean():.3f}")


## 9 · FAISS index

Builds an exact inner product index over the combined embeddings.
Self-retrieval test: the first product should find itself at rank 1 with score 1.0.


In [ ]:
index = faiss.IndexFlatIP(DIM)
index.add(combined_emb)
print(f"FAISS index — {index.ntotal:,} vectors  |  dim={DIM}")

D, I = index.search(combined_emb[0:1], k=6)
print(f"\nSelf-retrieval test — '{products['product_title'].iloc[0][:55]}...'")
for rank, (score, idx) in enumerate(zip(D[0], I[0])):
    marker = "  <- itself" if idx == 0 else ""
    print(f"  {rank+1}. [{score:.4f}]  {str(products['product_title'].iloc[idx])[:55]}{marker}")


## 10 · Save output files

Saves embeddings as numpy arrays and the FAISS index as a binary file.
Also saves `embedding_index.csv` which maps each FAISS row to product metadata.


In [ ]:
np.save(os.path.join(DATA_DIR, "product_embeddings.npy"),  product_emb)
np.save(os.path.join(DATA_DIR, "review_embeddings.npy"),   review_emb)
np.save(os.path.join(DATA_DIR, "combined_embeddings.npy"), combined_emb)
faiss.write_index(index, os.path.join(DATA_DIR, "faiss_index.bin"))

# Salva anche alpha per prodotto — utile per analisi e debug
alpha_df = pd.DataFrame({
    "parent_asin": list(asin_alpha.keys()),
    "alpha":       list(asin_alpha.values())
})

bins   = [0, 10, 25, 50, 100, float("inf")]
labels = ["budget","low","mid","high","premium"]
index_df = products[["parent_asin","product_title","brand",
                       "product_avg_rating","product_rating_count","price"]].copy()
index_df = index_df.merge(alpha_df, on="parent_asin", how="left")
index_df["emb_row"]      = range(len(index_df))
index_df["price_bucket"] = pd.cut(index_df["price"], bins=bins, labels=labels)
index_df.to_csv(os.path.join(DATA_DIR, "embedding_index.csv"), index=False)

print("Files saved:")
for fname in ["product_embeddings.npy","review_embeddings.npy",
               "combined_embeddings.npy","faiss_index.bin","embedding_index.csv"]:
    p = os.path.join(DATA_DIR, fname)
    print(f"  {fname:<38}  {os.path.getsize(p)/1e6:.1f} MB")


## 11 · Search test

Quick end-to-end test of the retrieval pipeline.
Use this to verify that the embeddings are working before running notebook 04.


In [ ]:
index_df_loaded = pd.read_csv(os.path.join(DATA_DIR, "embedding_index.csv"))

def search(query, k=5, price_bucket=None):
    q_vec  = model.encode([query], normalize_embeddings=True,
                           convert_to_numpy=True).astype(np.float32)
    n_cand = k * 10 if price_bucket else k
    D, I   = index.search(q_vec, n_cand)
    res    = index_df_loaded.iloc[I[0]].copy()
    res["score"] = D[0]
    if price_bucket:
        res = res[res["price_bucket"] == price_bucket]
    return res.head(k)[["product_title","brand","price","price_bucket",
                          "product_avg_rating","alpha","score"]]

queries = [
    ("affordable moisturizer for sensitive skin", None),
    ("air purifier quiet bedroom allergies",       None),
    ("cheap sunscreen SPF 50",                    "budget"),
    ("vitamin C supplement immune system",         None),
    ("electric toothbrush whitening",              None),
]

for query, bucket in queries:
    filtro = f"  [price: {bucket}]" if bucket else ""
    print(f"\n{'─'*68}")
    print(f"Query: '{query}'{filtro}")
    print('─'*68)
    res = search(query, k=5, price_bucket=bucket)
    for i, (_, row) in enumerate(res.iterrows()):
        prezzo = f"${row['price']:.2f}" if pd.notna(row['price']) else "  N/D"
        alpha  = f"a={row['alpha']:.2f}" if pd.notna(row.get('alpha')) else ""
        titolo = str(row["product_title"])[:50]
        print(f"  {i+1}. [{row['score']:.4f}] ⭐{row['product_avg_rating']:.1f}"
              f"  {prezzo:>8}  {alpha}  {titolo}")
